In [1]:
import numpy as np

class CoupledLSTM:
    def __init__(self, input_size, hidden_size):
        """
        Initializes the Coupled Input-Forget Gate LSTM network.
        In this variant, the forget gate is coupled to the input gate (f_t = 1 - i_t).
        Therefore, the parameters for the forget gate are omitted.
        """
        self.input_size = input_size
        self.hidden_size = hidden_size

        # Total size of concatenated input and hidden state
        concat_size = input_size + hidden_size

        # Random initialization of weights (scaled by 0.1 for stability) and zero biases

        # Weights and biases for the Input Gate (i_t)
        self.W_i = np.random.randn(hidden_size, concat_size) * 0.1
        self.b_i = np.zeros((hidden_size, 1))

        # Weights and biases for the Cell Candidate (C_tilde_t)
        self.W_c = np.random.randn(hidden_size, concat_size) * 0.1
        self.b_c = np.zeros((hidden_size, 1))

        # Weights and biases for the Output Gate (o_t)
        self.W_o = np.random.randn(hidden_size, concat_size) * 0.1
        self.b_o = np.zeros((hidden_size, 1))

    def _sigmoid(self, x):
        """
        Applies the sigmoid activation function.
        Clipping is used to prevent Overflow errors during exponentiation.
        """
        x = np.clip(x, -500, 500)
        return 1.0 / (1.0 + np.exp(-x))

    def forward_step(self, x_t, h_prev, c_prev):
        """
        Executes a single time step of the Coupled LSTM.
        """
        # Concatenate the current input and previous hidden state vertically
        # x_t shape: (input_size, 1), h_prev shape: (hidden_size, 1)
        concat_input = np.vstack((h_prev, x_t))

        # 1. Calculate the input gate (i_t)
        i_t = self._sigmoid(np.dot(self.W_i, concat_input) + self.b_i)

        # 2. Calculate the new cell candidate (C_tilde_t)
        c_tilde_t = np.tanh(np.dot(self.W_c, concat_input) + self.b_c)

        # 3. Update the cell state (C_t) using the coupled equation
        # Note: The forget gate operation is implicitly (1 - i_t)
        c_t = (1 - i_t) * c_prev + i_t * c_tilde_t

        # 4. Calculate the output gate (o_t)
        o_t = self._sigmoid(np.dot(self.W_o, concat_input) + self.b_o)

        # 5. Calculate the new hidden state (h_t)
        h_t = o_t * np.tanh(c_t)

        return h_t, c_t

    def forward(self, X):
        """
        Executes the full Forward Pass over an entire sequence.
        X shape expected: (sequence_length, input_size)
        """
        seq_length, input_features = X.shape
        if input_features != self.input_size:
            raise ValueError("Input feature dimension does not match the initialized input_size.")

        # Initialize hidden state and cell state with zeros for the first time step
        h_t = np.zeros((self.hidden_size, 1))
        c_t = np.zeros((self.hidden_size, 1))

        # List to collect the hidden states at each time step
        hidden_states = []

        # Iterate sequentially over the time steps
        for t in range(seq_length):
            # Extract the input for the current time step and reshape it to a column vector
            x_t = X[t].reshape(-1, 1)

            # Execute one forward step
            h_t, c_t = self.forward_step(x_t, h_t, c_t)

            # Store the computed hidden state
            hidden_states.append(h_t)

        # Convert the list to a NumPy array and remove unnecessary dimensions
        # Final shape: (sequence_length, hidden_size)
        return np.array(hidden_states).squeeze()


In [2]:
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# 1. Define dimensions based on the problem description
SEQ_LENGTH = 50      # 50 time steps as explicitly requested in the prompt
INPUT_SIZE = 12      # Arbitrary feature size for the synthetic input
HIDDEN_SIZE = 16     # Arbitrary hidden state size

print("Generating Synthetic Time Series...")
# 2. Generate Synthetic Time Series data
# Shape: (Sequence Length, Input Size) -> (50, 12)
synthetic_series = np.random.randn(SEQ_LENGTH, INPUT_SIZE)

# 3. Instantiate the model (Weights are initialized randomly inside the class)
lstm_model = CoupledLSTM(input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE)

# 4. Apply the model to the synthetic data to extract h_t for all time steps
# The forward method returns an array of hidden states for each time step
all_h_t = lstm_model.forward(synthetic_series)

# 5. Print results to verify matrix dimensions match perfectly
print("\n--- Matrix Dimensions Verification ---")
print(f"Input Sequence Shape: {synthetic_series.shape} -> (Time Steps, Features)")
print(f"Extracted h_t Shape:  {all_h_t.shape} -> (Time Steps, Hidden Size)")

print("\n--- Extracted Hidden States (h_t) Preview ---")
print("Showing h_t for the first time step (t=1):")
print(all_h_t[0])
print("\n...")
print("\nShowing h_t for the last time step (t=50):")
print(all_h_t[-1])


Generating Synthetic Time Series...

--- Matrix Dimensions Verification ---
Input Sequence Shape: (50, 12) -> (Time Steps, Features)
Extracted h_t Shape:  (50, 16) -> (Time Steps, Hidden Size)

--- Extracted Hidden States (h_t) Preview ---
Showing h_t for the first time step (t=1):
[ 0.00756655 -0.00179809 -0.03225685  0.03710601 -0.05024182 -0.09646721
  0.10374103 -0.02787929  0.09012015 -0.04885495 -0.02104314 -0.01749433
 -0.02297694  0.03677362  0.03973274 -0.10707274]

...

Showing h_t for the last time step (t=50):
[ 0.07996865  0.06500008 -0.02595329  0.16411744 -0.05881969 -0.07114629
  0.02801032 -0.05714587 -0.03435506  0.02720256  0.0381528  -0.03984996
 -0.07629688 -0.0388831  -0.07059517 -0.00919359]
